# 08 — Interpretability for Safety

Using mechanistic interpretability to detect dangerous capabilities.

## Detecting Dangerous Capabilities via Activation Analysis

Interpretability tools (SAEs, activation patching, linear probes) can be used to detect safety-relevant features in model activations:

**Capability detection**: does the model have internal representations for dangerous tasks?
- Probe for 'bioweapon synthesis knowledge' directions in activation space
- Probe for 'deceptive intent' vs 'honest intent' directions
- Look for features active on jailbreak prompts that are absent on normal prompts

**Honesty probes** (Marks & Tegmark, 2023): train a linear classifier to distinguish 'model is reporting what it believes' from 'model is saying what will be rewarded'. Truthful statements and lies may activate different directions even when the model suppresses visible differences.

**Safety steering vectors**: add a 'harmlessness direction' to activations to prevent harmful outputs without fine-tuning.

**Limitations**: probe results depend heavily on the training distribution for the probe. A model could learn to represent beliefs in ways that defeat linear probes (superposition, distributed coding).

In [ ]:
# Capability detection via activation analysis
import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

torch.manual_seed(42)

# Simulate a language model with hidden activations
class SafetyAwareLM(nn.Module):
    def __init__(self, d_model=64, vocab=100):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.layer1 = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU())
        self.layer2 = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU())
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x):
        h = self.embed(x).mean(1)  # mean pooling over sequence
        h1 = self.layer1(h)
        h2 = self.layer2(h1)
        return self.head(h2), {'layer1': h1, 'layer2': h2}

model = SafetyAwareLM()

# Simulate two types of prompts
n_per_class = 200

# Harmful prompts: start with token 0 (proxy for harmful intent)
harmful_tokens = torch.randint(1, 10, (n_per_class, 16))  # low token IDs = 'harmful'
harmful_tokens[:, 0] = 0

# Benign prompts: higher token IDs
benign_tokens = torch.randint(50, 100, (n_per_class, 16))

# Get activations
with torch.no_grad():
    _, harmful_acts = model(harmful_tokens)
    _, benign_acts = model(benign_tokens)

# Train safety probes on each layer
for layer_name in ['layer1', 'layer2']:
    harm_feat = harmful_acts[layer_name].numpy()
    benign_feat = benign_acts[layer_name].numpy()

    X_probe = np.vstack([harm_feat, benign_feat])
    y_probe = np.array([1]*n_per_class + [0]*n_per_class)  # 1=harmful, 0=benign

    probe = LogisticRegression(max_iter=500)
    scores = cross_val_score(probe, X_probe, y_probe, cv=5)
    print(f'{layer_name} safety probe: {scores.mean():.3f} ± {scores.std():.3f}')

# Interpret the probe direction
probe.fit(X_probe, y_probe)
direction = probe.coef_[0]
top_features = np.argsort(np.abs(direction))[::-1][:5]
print(f'Top safety-relevant dimensions: {top_features}')
print('(These dimensions best separate harmful from benign prompts)')